# Tool Use 기초 (4) — 완성된 리마인더 시스템

**Skilljar Lesson 09 대응**

이 노트북에서 다루는 내용:
1. 3개 도구를 모두 등록하여 완전한 Reminder System 구성
2. `run_tool()` 라우팅 업데이트
3. 복합 시나리오 테스트 (날짜 계산 + 리마인더 설정)
4. 대화 히스토리 분석

**3 Tools**:
- `get_current_datetime` — 현재 날짜/시간 조회
- `add_duration_to_datetime` — 날짜에 기간 더하기
- `set_reminder` — 리마인더 등록

In [ ]:
# ── Setup ──────────────────────────────────────────────
import anthropic
from datetime import datetime, timedelta
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

In [ ]:
# ── All 3 Tool Functions ──────────────────────────────

def get_current_datetime(date_format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """현재 날짜/시간을 지정된 포맷으로 반환합니다."""
    valid_formats = [
        "%Y-%m-%d %H:%M:%S", "%Y-%m-%d", "%H:%M:%S",
        "%H:%M", "%Y/%m/%d", "%m/%d/%Y",
    ]
    if date_format not in valid_formats:
        raise ValueError(
            f"Invalid date format: {date_format}. Valid formats: {valid_formats}"
        )
    return datetime.now().strftime(date_format)


def add_duration_to_datetime(
    date_str: str,
    duration_days: int = 0,
    duration_hours: int = 0,
    duration_minutes: int = 0,
) -> str:
    """주어진 날짜/시간에 기간을 더합니다."""
    if not date_str:
        raise ValueError("date_str cannot be empty")
    dt = datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S")
    dt += timedelta(
        days=duration_days,
        hours=duration_hours,
        minutes=duration_minutes,
    )
    return dt.strftime("%Y-%m-%d %H:%M:%S")


# 리마인더 저장소
reminders = []


def set_reminder(reminder_text: str, reminder_datetime: str) -> str:
    """
    리마인더를 등록합니다.

    Args:
        reminder_text: 리마인더 내용
        reminder_datetime: 리마인더 날짜/시간 문자열

    Returns:
        확인 메시지
    """
    if not reminder_text:
        raise ValueError("reminder_text cannot be empty")
    reminder = {"text": reminder_text, "datetime": reminder_datetime}
    reminders.append(reminder)
    return f"Reminder set: '{reminder_text}' at {reminder_datetime}"

In [ ]:
# ── All 3 Tool Schemas ─────────────────────────────────

get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": (
        "Returns the current date and time in a specified format. "
        "Defaults to '%Y-%m-%d %H:%M:%S'."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": (
                    "The format string for the date/time output. "
                    "Supported: '%Y-%m-%d %H:%M:%S', '%Y-%m-%d', '%H:%M:%S', "
                    "'%H:%M', '%Y/%m/%d', '%m/%d/%Y'."
                ),
                "default": "%Y-%m-%d %H:%M:%S",
            }
        },
        "required": [],
    },
}

add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": (
        "Adds a duration (days, hours, minutes) to a given datetime string. "
        "Input datetime must be in '%Y-%m-%d %H:%M:%S' format."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "date_str": {
                "type": "string",
                "description": "The starting datetime in '%Y-%m-%d %H:%M:%S' format.",
            },
            "duration_days": {
                "type": "integer",
                "description": "Number of days to add. Defaults to 0.",
                "default": 0,
            },
            "duration_hours": {
                "type": "integer",
                "description": "Number of hours to add. Defaults to 0.",
                "default": 0,
            },
            "duration_minutes": {
                "type": "integer",
                "description": "Number of minutes to add. Defaults to 0.",
                "default": 0,
            },
        },
        "required": ["date_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": (
        "Sets a reminder with a text description and a specific datetime. "
        "Use this to help users create reminders for future events."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "reminder_text": {
                "type": "string",
                "description": "The text/description of the reminder.",
            },
            "reminder_datetime": {
                "type": "string",
                "description": (
                    "The datetime for the reminder. "
                    "Preferably in '%Y-%m-%d %H:%M:%S' format."
                ),
            },
        },
        "required": ["reminder_text", "reminder_datetime"],
    },
}

# 모든 도구 등록
all_tools = [
    get_current_datetime_schema,
    add_duration_to_datetime_schema,
    set_reminder_schema,
]

print(f"Registered {len(all_tools)} tools:")
for t in all_tools:
    print(f"  - {t['name']}")

In [ ]:
# ── Helper Functions (Notebook 03에서 가져옴) ──────────

def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})


def add_assistant_message(messages, response):
    if isinstance(response, anthropic.types.Message):
        messages.append({"role": "assistant", "content": response.content})
    else:
        messages.append({"role": "assistant", "content": response})


def chat(messages, tools=None, system=None):
    kwargs = {
        "model": MODEL,
        "max_tokens": 4096,
        "messages": messages,
    }
    if tools:
        kwargs["tools"] = tools
    if system:
        kwargs["system"] = system
    return client.messages.create(**kwargs)


def text_from_message(response):
    texts = []
    for block in response.content:
        if block.type == "text":
            texts.append(block.text)
    return "\n".join(texts)

## §1. run_tool() 업데이트 — 3개 도구 라우팅 (Updated Routing)

`set_reminder`를 추가하여 모든 도구를 라우팅합니다.

In [ ]:
def run_tool(tool_name, tool_input):
    """도구 이름으로 해당 함수를 실행합니다."""
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    else:
        return f"Error: Unknown tool '{tool_name}'"


def run_tools(response):
    """응답의 모든 tool_use 블록을 실행합니다."""
    tool_results = []
    for block in response.content:
        if block.type != "tool_use":
            continue

        tool_name = block.name
        tool_input = block.input
        print(f"  🔧 Running tool: {tool_name}({tool_input})")

        try:
            result = run_tool(tool_name, tool_input)
            print(f"  ✅ Result: {result}")
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": str(result),
            })
        except Exception as e:
            error_msg = f"Error executing {tool_name}: {str(e)}"
            print(f"  ❌ {error_msg}")
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": error_msg,
                "is_error": True,
            })

    return tool_results


def run_conversation(user_message, tools, system=None, verbose=True):
    """사용자 메시지로 시작하여 자동 멀티턴 대화를 실행합니다."""
    messages = []
    add_user_message(messages, user_message)

    if verbose:
        print(f"User: {user_message}")
        print("=" * 60)

    turn = 0
    while True:
        turn += 1
        if verbose:
            print(f"\n--- Turn {turn} ---")

        response = chat(messages, tools=tools, system=system)

        if verbose:
            print(f"stop_reason: {response.stop_reason}")

        if response.stop_reason != "tool_use":
            if verbose:
                print(f"\nClaude: {text_from_message(response)}")
            return response, messages

        add_assistant_message(messages, response)
        tool_results = run_tools(response)
        messages.append({"role": "user", "content": tool_results})

    return response, messages

## §2. 테스트: 복합 시나리오 — 의사 예약 리마인더

"Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050."

Claude가 수행할 단계:
1. `add_duration_to_datetime` — 2050-01-01에서 177일 후 계산
2. `set_reminder` — 계산된 날짜에 리마인더 등록

In [ ]:
reminders.clear()  # 리마인더 초기화

response, messages = run_conversation(
    "Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050.",
    tools=all_tools,
)

## §3. 테스트: 현재 시간 기반 리마인더

"Remind me to call mom in 3 hours"

Claude가 수행할 단계:
1. `get_current_datetime` — 현재 시간 확인
2. `add_duration_to_datetime` — 3시간 후 계산
3. `set_reminder` — 리마인더 등록

In [ ]:
response, messages = run_conversation(
    "Remind me to call mom in 3 hours",
    tools=all_tools,
)

## §4. 등록된 리마인더 확인 (Examining Reminders)

Python 리스트에 저장된 모든 리마인더를 확인합니다.

In [ ]:
print(f"Total reminders: {len(reminders)}")
print()
for i, r in enumerate(reminders, 1):
    print(f"Reminder {i}:")
    print(f"  Text:     {r['text']}")
    print(f"  Datetime: {r['datetime']}")
    print()

## §5. 대화 히스토리 분석 (Examining Conversation History)

마지막 대화의 messages 구조를 살펴봅니다.  
멀티턴 Tool Use가 어떻게 메시지 배열로 표현되는지 확인할 수 있습니다.

In [ ]:
print(f"Total messages in last conversation: {len(messages)}")
print()

for i, msg in enumerate(messages):
    role = msg["role"]
    content = msg["content"]

    if isinstance(content, str):
        print(f"[{i}] {role}: {content[:80]}..." if len(content) > 80 else f"[{i}] {role}: {content}")
    elif isinstance(content, list):
        block_types = []
        for block in content:
            if isinstance(block, dict):
                block_types.append(block.get("type", "unknown"))
            else:
                block_types.append(getattr(block, "type", "unknown"))
        print(f"[{i}] {role}: [{', '.join(block_types)}]")
    else:
        print(f"[{i}] {role}: (complex content)")